# Getting started

This is the first notebook in the series. It checks that the package imports, that a date can be
written three ways, and that holiday calendars behave. Later notebooks assume this setup and
start the same way: Python's logging is switched on, and so is the library logger.

From the repository root, once:

```
python -m pip install -e ".[front]" matplotlib
```

`front` installs pandas. matplotlib draws the charts in the next notebooks. The install compiles
`_core`. Shut the kernel down before installing again, so Windows can replace the extension.


In [ ]:
import logging
from datetime import date

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import sjd

sjd.set_log_level("INFO")

info = sjd.build_info()
print("version", sjd.__version__)
for key in ("compiler", "build_type", "quantlib", "boost", "git_commit"):
    print(f"{key:12} {info[key]}")

assert sjd.__version__ == info["version"]
assert info["build_type"]
assert info["compiler"]
assert isinstance(info["git_commit"], str)


## Dates

A date argument accepts a `datetime.date`, a `numpy.datetime64`, or a `pandas.Timestamp`.
QuantLib dates stay inside the library. Christmas 2026 is a Friday, so the following business
day is Tuesday 29 December, after the weekend and the Boxing Day holiday.


In [ ]:
import numpy as np
import pandas as pd

london = sjd.Calendar("UnitedKingdom")
christmas = date(2026, 12, 25)
rolled = london.adjust(christmas)

assert london.adjust(np.datetime64("2026-12-25")) == rolled
assert london.adjust(pd.Timestamp(christmas)) == rolled
assert rolled == date(2026, 12, 29)
print(christmas, "rolls to", rolled)


## Calendars

`Calendar.names()` lists the QuantLib calendars the constructor accepts. A calendar with
components is closed when any component is closed. USDZAR, the calendar on the sample curve,
is closed on a United States holiday and on a South African holiday.

`adjust` rolls a holiday onto a business day. `advance` moves a number of business days.
`business_days_between` counts them, including the first date and excluding the last.
`holidays` lists the dates in between.


In [ ]:
names = sjd.Calendar.names()
assert "UnitedKingdom" in names
assert "UnitedKingdom.Metals" in names

assert london.name == "UnitedKingdom"
assert london.components == ()
assert london.is_business_day(date(2026, 12, 24))
assert london.is_holiday(christmas)
assert london.adjust(christmas, sjd.BusinessDayConvention.PRECEDING) == date(2026, 12, 24)
assert london.holidays(date(2026, 12, 1), date(2026, 12, 31)) == [
    date(2026, 12, 25),
    date(2026, 12, 28),
]
assert london.business_days_between(date(2026, 12, 21), date(2026, 12, 31)) == 6
assert london.advance(
    date(2026, 11, 30),
    1,
    sjd.TimeUnit.MONTHS,
    sjd.BusinessDayConvention.MODIFIED_FOLLOWING,
) == date(2026, 12, 30)

usdzar = sjd.Calendar("USDZAR", ["UnitedStates", "SouthAfrica"])
independence_day = date(2026, 7, 3)
assert usdzar.components == ("UnitedStates", "SouthAfrica")
assert sjd.Calendar("SouthAfrica").is_business_day(independence_day)
assert not usdzar.is_business_day(independence_day)
print(usdzar)
print("closed on", independence_day, "because the United States is closed")


## The sample dates

The curve used from here on is anchored on 16 September 2026. The collar settles three months
later, rolled to the next USDZAR business day, and expires two business days before that.
These are the dates `sjd.examples.sample_market` uses.


In [ ]:
from sjd import examples

anchor = examples.ANCHOR
holidays = examples.sample_calendar()
collar_settlement = holidays.adjust(examples.add_months(anchor, 3))
collar_maturity = holidays.advance(collar_settlement, -2)
market = examples.sample_market()

assert holidays.is_business_day(collar_settlement)
assert collar_settlement == market["collar_settlement"]
assert collar_maturity == market["collar_maturity"]
print("anchor   ", anchor)
print("maturity ", collar_maturity)
print("settle   ", collar_settlement)


## What can go wrong

An unknown calendar name cannot be constructed. `ConfigurationError` is a `ValueError`, and
`.code` is the stable identifier `CONFIGURATION`.


In [ ]:
try:
    sjd.Calendar("Narnia")
except sjd.ConfigurationError as error:
    assert isinstance(error, ValueError)
    assert isinstance(error, sjd.SjdError)
    assert error.code == "CONFIGURATION"
    assert "Narnia" in str(error)
    print(type(error).__name__, error.code)
    print(error)
else:
    raise AssertionError("an unknown calendar name should be refused")


## What it logs

`logging.basicConfig` shows records from the root logger. `sjd.set_log_level("INFO")` sets the
C++ logger and the `sjd` logger together. C++ records are written to `sjd._core`. Nothing in
this notebook asks the library to log. The curve notebooks produce the first warnings.


In [ ]:
class Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []

    def emit(self, record):
        self.records.append(record)

captured = Collect()
logging.getLogger("sjd._core").addHandler(captured)
sjd.Calendar("UnitedKingdom")

assert logging.getLogger("sjd").level == logging.INFO
assert captured.records == []
print("sjd logger level: INFO")
print("records from building a calendar:", len(captured.records))
